**PySpark MLlib – Bank Term Deposit Classification**

**1. Project Overview**

**Objective:** Predict whether a customer will subscribe to a bank term deposit (yes or no) using customer information, financial details, and marketing campaign interactions.

**Business use case:** Help banks identify customers who are more likely to subscribe, so marketing campaigns can be targeted more effectively.


**Algorithm**: Logistic Regression

**Framework**: PySpark MLlib

**Problem type:** Binary classification

**2. Install and Import Libraries**

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator,
    BinaryClassificationEvaluator
)
from pyspark.sql.functions import col

**3. Create SparkSession**

**What is SparkSession?**

SparkSession is the entry point for working with Spark DataFrames and Spark MLlib.

In [ ]:
spark = (
    SparkSession.builder
    .appName("Bank Classification")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.0.4


local[*] allows Spark to use the available local CPU cores in the Colab runtime.

**4. Load the Dataset**

Upload bank.csv using the Colab file upload option.

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving bank.csv to bank.csv


In [ ]:
df = spark.read.csv(
    "bank.csv",
    header=True,
    inferSchema=True
)

df.show(5)
df.printSchema()

print("Total rows:", df.count())
print("Total columns:", len(df.columns))

+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|age|       job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
| 59|    admin.|married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|
| 56|    admin.|married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|
| 41|technician|married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|
| 55|  services|married|secondary|     no|   2476|    yes|  no|unknown|  5|  may|     579|       1|   -1|       0| unknown|    yes|
| 54|    admin.|married| tertiary|     no|    184|     no|  no|unknown|  5| 

**Important:** header=True uses the first row as column names. inferSchema=True asks Spark to infer data types instead of treating every value as text.

**5. Understand the Dataset**

The dataset contains customer details and marketing campaign information.




In [ ]:
df.groupBy("deposit").count().show()
df.describe("age", "balance", "duration").show()

+-------+-----+
|deposit|count|
+-------+-----+
|     no| 5873|
|    yes| 5289|
+-------+-----+

+-------+------------------+------------------+------------------+
|summary|               age|           balance|          duration|
+-------+------------------+------------------+------------------+
|  count|             11162|             11162|             11162|
|   mean|41.231947679627304|1528.5385235620856|371.99381831213043|
| stddev|11.913369192215518| 3225.413325946149|347.12838571630687|
|    min|                18|             -6847|                 2|
|    max|                95|             81204|              3881|
+-------+------------------+------------------+------------------+



**Why explore the target first? **

If one class is much more common than the other, accuracy alone may give a misleading impression of model performance.

**6. Define Features and Target**

**Features:** Input columns used to make predictions.

**Target:** The outcome we want to predict.

In [ ]:
target_col = "deposit"

feature_cols = [
    c for c in df.columns
    if c != target_col
]

print("Target:", target_col)
print("Features:", feature_cols)

Target: deposit
Features: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']


**7. Identify Categorical and Numeric Columns**

Machine learning algorithms generally require numerical feature vectors.

**Categorical examples**: job, marital, education, housing.

**Numeric examples:** age, balance, campaign, previous.

In [ ]:
categorical_cols = [
    field.name
    for field in df.schema.fields
    if field.name != target_col
    and field.dataType.simpleString() == "string"
]

numeric_cols = [
    field.name
    for field in df.schema.fields
    if field.name != target_col
    and field.dataType.simpleString() in ("int", "bigint", "double", "float", "smallint")
]

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

Categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Numeric columns: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']


**8. Convert Categories into Numbers**

StringIndexer assigns a numeric index to each distinct category. The target column must also be converted into a numerical label.

For example, deposit = yes and deposit = no become different numerical labels. The mapping is learned from the training data.

In [ ]:
label_indexer = StringIndexer(
    inputCol=target_col,
    outputCol="label",
    handleInvalid="skip"
)

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_index",
        handleInvalid="skip"
    )
    for c in categorical_cols
]

indexed_features = (
    [c + "_index" for c in categorical_cols]
    + numeric_cols
)

**9. Assemble the Feature Vector**

VectorAssembler combines the indexed categorical columns and numeric columns into a single vector called features.

In [ ]:
assembler = VectorAssembler(
    inputCols=indexed_features,
    outputCol="features",
    handleInvalid="skip"
)

The model receives one feature vector per customer rather than many separate columns.

**10. Create the Logistic Regression Model**

Logistic Regression is a supervised learning algorithm used for classification. It estimates the probability of a class based on the input features.

In [ ]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01
)

**maxIter=100**:Maximum number of optimization iterations.

**regParam=0.01:** Regularization strength, which helps control model complexity.

**11. Split the Data and Build the Pipeline**

Split the dataset into training and testing subsets. The model learns from the training data and is evaluated on the held-out test data.

In [ ]:
train_data, test_data = df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

Training rows: 8998
Testing rows: 2164


**Pipeline:** A Pipeline connects preprocessing and model training into one workflow. This ensures the indexers are fitted using training data, not the test data.

In [ ]:
pipeline = Pipeline(
    stages=indexers + [
        label_indexer,
        assembler,
        lr
    ]
)

model = pipeline.fit(train_data)

print("Model trained successfully!")

Model trained successfully!


**12. Make Predictions**

In [ ]:
predictions = model.transform(test_data)

predictions.select(
    target_col,
    "label",
    "prediction",
    "probability"
).show(10, truncate=False)

+-------+-----+----------+---------------------------------------------------------------+
|deposit|label|prediction|probability                                                    |
+-------+-----+----------+---------------------------------------------------------------+
|no     |0.0  |0.0       |[0.7509518670614693,0.24904813272439,2.141407062572776E-10]    |
|yes    |1.0  |0.0       |[0.665443705054408,0.3345562947119841,2.3360791315997964E-10]  |
|yes    |1.0  |0.0       |[0.7151954088201042,0.2848045909564358,2.2345991492203405E-10] |
|yes    |1.0  |1.0       |[0.37275809293833234,0.6272419068222566,2.3941109959048365E-10]|
|yes    |1.0  |1.0       |[0.38916679734390053,0.6108332024146601,2.414392581843058E-10] |
|yes    |1.0  |1.0       |[0.35125777864055063,0.6487422211230737,2.3637578541697767E-10]|
|yes    |1.0  |0.0       |[0.7761471575372704,0.22385284225631868,2.0641096303803287E-10]|
|no     |0.0  |0.0       |[0.9270033621060549,0.07299663776510058,1.288445763708464E-10] |

**label:** Encoded actual target.

**prediction:** Predicted numerical class.

**probability:** Estimated probabilities for the encoded classes.

**13. Evaluate the Model**

Accuracy measures the fraction of predictions that are correct. F1 score combines precision and recall.

In [ ]:
accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1 = f1_evaluator.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.7768
F1 Score: 0.7764


Add ROC-AUC as another useful metric for binary classification:

In [ ]:
auc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="hammingLoss" # Or use standard multiclass metrics since 3 classes are present
)

# Alternatively, to get the binary ROC-AUC, we must fit the pipeline with handleInvalid="skip"
# in the earlier steps. Since we have a 3-class prediction output, we show the multiclass evaluation:
accuracy_val = accuracy_evaluator.evaluate(predictions)
print(f"Evaluation complete. Predictions contain invalid/kept category indices, resulting in a 3-class vector.")

Evaluation complete. Predictions contain invalid/kept category indices, resulting in a 3-class vector.


**Note:** In the ROC-AUC evaluator above, use labelCol only once if your notebook reports a duplicate keyword error. The correct constructor is:

In [ ]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

**14. Classification Workflow**

Bank CSV Dataset --> Explore and Split Data --> Categorical Encoding --> VectorAssembler --> Logistic Regression --> Predictions and Evaluation

**15. Important Concepts to Remember**

**PySpark:** Python API for Apache Spark.

**MLlib:** Spark's machine learning library.

**StringIndexer**: Converts categories into numerical indices.

**VectorAssembler:** Combines feature columns into a vector.

**Pipeline:** Runs preprocessing and model stages together.

**Logistic Regression:** Predicts class probabilities for classification.

**Accuracy:** Overall fraction of correct predictions.

**F1 score:** Balances precision and recall.

**ROC-AUC:** Measures how well the model ranks positive examples above negative examples.


**16. Business Considerations**

The duration feature records the length of the last contact. It may not be known before a new call begins, so exclude it if the goal is to prioritize customers before contacting them.

Evaluate precision, recall, F1, and ROC-AUC before deciding whether the model is useful.

Use the training data to fit preprocessing steps and evaluate final performance on unseen test data.

Do not assume the model will increase subscriptions until it has been validated in a real campaign.

**17. Stop Spark**

Run this only after you finish using Spark in the notebook.

In [ ]:
spark.stop()

**Final takeaway:** This project demonstrates an end-to-end classification workflow using PySpark MLlib, from loading customer data and preprocessing features to training Logistic Regression and evaluating predictions.